# Predictive Analytics Using Historical Data — Daily Delhi Climate

This notebook cleans historical weather data (`DailyDelhiClimateTest.csv`), engineers lag and rolling features, compares **Linear Regression**, **Random Forest**, and **SARIMA** models, evaluates accuracy (**MAE, RMSE, MAPE, R²**), and visualizes future temperature trends.

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

## 1. Load Historical Dataset
If running in **Google Colab** and `DailyDelhiClimateTest.csv` is not already in the folder, a file upload button will appear automatically.

In [ ]:
csv_filename = 'DailyDelhiClimateTest.csv'
if not Path(csv_filename).exists():
    try:
        from google.colab import files
        print('Please upload DailyDelhiClimateTest.csv:')
        uploaded = files.upload()
        csv_filename = next(iter(uploaded.keys()))
    except ImportError:
        raise FileNotFoundError(f'Place {csv_filename} in the notebook directory.')

raw_df = pd.read_csv(csv_filename)
print('Dataset shape:', raw_df.shape)
raw_df.head()

## 2. Clean & Preprocess Historical Data
- Parse `date`, remove duplicates, sort chronologically, and set daily frequency.
- Detect and interpolate extreme pressure outliers (for example `meanpressure = 59.0` on `2017-01-01`).

In [ ]:
df = raw_df.copy()
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df.dropna(subset=['date']).drop_duplicates(subset=['date'])
df = df.sort_values('date').set_index('date').asfreq('D')

numeric_cols = ['meantemp', 'humidity', 'wind_speed', 'meanpressure']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

q1, q3 = df['meanpressure'].quantile([0.25, 0.75])
iqr = q3 - q1
outliers = (df['meanpressure'] < q1 - 3 * iqr) | (df['meanpressure'] > q3 + 3 * iqr)
print('Pressure outliers detected:', int(outliers.sum()))
df.loc[outliers, 'meanpressure'] = np.nan

df[numeric_cols] = df[numeric_cols].interpolate(method='time').bfill().ffill()
df.describe().round(2)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True)
for ax, (col, title, color) in zip(
    axes.flat,
    [
        ('meantemp', 'Mean Temperature (°C)', 'tab:red'),
        ('humidity', 'Humidity (%)', 'tab:blue'),
        ('wind_speed', 'Wind Speed (km/h)', 'tab:green'),
        ('meanpressure', 'Mean Pressure (hPa)', 'tab:purple'),
    ],
):
    ax.plot(df.index, df[col], color=color, linewidth=1.8)
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=30)
plt.suptitle('Historical Climate Trends (Cleaned)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('01_historical_trends.png', dpi=150)
plt.show()

## 3. Feature Engineering & Chronological Train/Test Split
We use strictly past observations (`shift`) for lag and rolling features to prevent data leakage.

In [ ]:
feat_df = df.copy()
for lag in (1, 2, 3, 7):
    feat_df[f'temp_lag_{lag}'] = feat_df['meantemp'].shift(lag)
feat_df['temp_roll_mean_7'] = feat_df['meantemp'].shift(1).rolling(7).mean()
feat_df['temp_roll_std_7'] = feat_df['meantemp'].shift(1).rolling(7).std()
feat_df['humidity_lag_1'] = feat_df['humidity'].shift(1)
feat_df['wind_lag_1'] = feat_df['wind_speed'].shift(1)
feat_df['pressure_lag_1'] = feat_df['meanpressure'].shift(1)
feat_df['day_of_week'] = feat_df.index.dayofweek
feat_df['day_of_year'] = feat_df.index.dayofyear
feat_df = feat_df.dropna()

feature_cols = [c for c in feat_df.columns if c not in df.columns]
split_idx = int(len(feat_df) * 0.8)
train_df = feat_df.iloc[:split_idx]
test_df = feat_df.iloc[split_idx:]

X_train, y_train = train_df[feature_cols], train_df['meantemp']
X_test, y_test = test_df[feature_cols], test_df['meantemp']
print(f'Train rows: {len(X_train)} | Test rows: {len(X_test)}')

## 4. Train Regression & Time-Series Models

In [ ]:
lr = LinearRegression().fit(X_train, y_train)
lr_pred = lr.predict(X_test)

rf = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42).fit(X_train, y_train)
rf_pred = rf.predict(X_test)

series_train = df.loc[:train_df.index[-1], 'meantemp']
sarima = SARIMAX(
    series_train,
    order=(1, 1, 1),
    seasonal_order=(1, 0, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False,
).fit(disp=False)
sarima_pred = sarima.forecast(steps=len(y_test)).values

preds = {
    'Linear Regression': lr_pred,
    'Random Forest': rf_pred,
    'SARIMA (1,1,1)(1,0,1,7)': sarima_pred,
}

## 5. Evaluate Accuracy & Visualize Predictions

In [ ]:
rows = []
for name, y_hat in preds.items():
    nz = y_test != 0
    rows.append({
        'Model': name,
        'MAE': mean_absolute_error(y_test, y_hat),
        'RMSE': float(np.sqrt(mean_squared_error(y_test, y_hat))),
        'MAPE (%)': float(np.mean(np.abs((y_test[nz] - y_hat[nz]) / y_test[nz])) * 100),
        'R2': r2_score(y_test, y_hat),
    })

metrics_df = pd.DataFrame(rows).sort_values('RMSE')
metrics_df.round(3)

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(y_test.index, y_test, label='Actual', color='black', linewidth=2.2)
for name, y_hat in preds.items():
    plt.plot(y_test.index, y_hat, linestyle='--', linewidth=1.8, label=name)
plt.title('Actual vs Predicted Mean Temperature (Test Set)')
plt.xlabel('Date')
plt.ylabel('Mean Temperature (°C)')
plt.legend()
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('02_actual_vs_predicted.png', dpi=150)
plt.show()

In [ ]:
best_name = metrics_df.iloc[0]['Model']
residuals = y_test.values - preds[best_name]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(y_test.index, residuals, marker='o', linestyle='-', color='tab:orange')
axes[0].axhline(0, color='black', linestyle='--')
axes[0].set_title(f'Residuals Over Time ({best_name})')
axes[0].tick_params(axis='x', rotation=30)
sns.histplot(residuals, kde=True, ax=axes[1], color='tab:orange')
axes[1].set_title(f'Residual Distribution ({best_name})')
plt.tight_layout()
plt.savefig('03_residual_analysis.png', dpi=150)
plt.show()

## 6. 14-Day Future Temperature Forecast

In [ ]:
full_sarima = SARIMAX(
    df['meantemp'],
    order=(1, 1, 1),
    seasonal_order=(1, 0, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False,
).fit(disp=False)
forecast_res = full_sarima.get_forecast(steps=14)
future_idx = pd.date_range(df.index[-1] + pd.Timedelta(days=1), periods=14, freq='D')
ci = forecast_res.conf_int()
future_df = pd.DataFrame({
    'forecast': forecast_res.predicted_mean.values,
    'lower_ci': ci.iloc[:, 0].values,
    'upper_ci': ci.iloc[:, 1].values,
}, index=future_idx)

plt.figure(figsize=(12, 5))
plt.plot(df.index, df['meantemp'], label='Historical Temperature', color='tab:blue')
plt.plot(future_df.index, future_df['forecast'], label='14-Day SARIMA Forecast', color='tab:red', linewidth=2)
plt.fill_between(future_df.index, future_df['lower_ci'], future_df['upper_ci'], color='tab:red', alpha=0.2, label='95% Confidence Interval')
plt.title('14-Day Future Temperature Forecast')
plt.xlabel('Date')
plt.ylabel('Mean Temperature (°C)')
plt.legend()
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('04_future_forecast.png', dpi=150)
plt.show()

future_df.round(2)